# 真实资产与数字孪生

输入 oven4 实拍 RGB-D、逐帧 Mask 与配套 USD，输出米制相机、增强深度、重建结果、配准及关节/部件指标。

**流程**：输入 → 相机与尺度 → 前景与深度 → 重建 → 配准 → 质量评估 → 导出。

本次研究烤箱门相对箱体的单轴旋转。start/end 表示采集顺序，并非机械限位标签；本例显式选择两个静态状态进行重建。


## 1. 输入与环境

核对实际解释器和所选方法，读取实拍与 USD。尺寸或命名冲突以显式选择的 USD 为准，并保留在报告中。

In [ ]:
import io
import json
import zipfile
from collections import Counter

from examples._display import (
    setup_notebook,
    show_animation,
    show_image_strip,
    show_metrics,
    show_reconstruction_evaluation,
    show_rgbd,
)
from PIL import Image

PROJECT_ROOT, OUTPUT_DIR = setup_notebook("real_assets_and_digital_twin")

from physrealx import ArtifactRef, MethodCapability, PhysRealX, ReconstructionEvaluationInput
from physrealx.integrations.registration import ReferenceRegistrationInput
from physrealx.integrations.rgbd import MetricRGBDInput
from physrealx.integrations.usd_reference import evaluate_reference_kinematics

px = PhysRealX.open(PROJECT_ROOT / ".physrealx")
for capability, method in (
    (MethodCapability.CAMERA_RECOVER, "colmap"),
    (MethodCapability.CAMERA_ALIGN_METRIC, "usd_depth_scale"),
    (MethodCapability.DEPTH_ENHANCE, "rgb_guided_fill"),
    (MethodCapability.GEOMETRY_RECONSTRUCT, "artgs"),
    (MethodCapability.CAMERA_ALIGN_METRIC, "articulated_surface_icp"),
):
    status = px.methods.inspect(capability, method)
    if not status.ready:
        raise RuntimeError(f"{method}: {status.detail}")


In [ ]:
inspection = px.assets.inspect_source(
    PROJECT_ROOT / "examples/real_assets_and_digital_twin/input/ArtModeling/oven4",
    importer="artmodeling",
    config={"states": ["can.start", "can.end"]},
)
if not inspection.ingestible:
    raise RuntimeError("ArtModeling 数据包未通过结构预检。")
observations = px.assets.ingest(
    PROJECT_ROOT / "examples/real_assets_and_digital_twin/input/ArtModeling/oven4",
    resource_id="oven4.observations",
    version="4.0.0",
    display_name="Oven 4 captures and reverse-engineered USD",
    importer="artmodeling",
    config={
        "states": ["can.start", "can.end"],
        "depth_unit": "native_u16_unscaled",
        "coordinate_frame": "camera_unregistered",
        "include_candidates": True,
        "view_alignment": {
            "can.start.rgb": {
                "mode": "fiducial_assisted",
                "fiducial": {"family": "aruco_6x6_50", "layout_id": "oven4.table.markers"},
            },
            "can.end.rgb": {
                "mode": "fiducial_assisted",
                "fiducial": {"family": "aruco_6x6_50", "layout_id": "oven4.table.markers"},
            },
        },
    },
)
reference_model = px.assets.import_reference(
    observations,
    usd_path="sim/usd/model_oven.usd",
    root_link="E_body_203",
    moving_links=("E_door_197",),
)
reference = json.loads(px.artifacts.get_bytes(reference_model))
show_metrics(
    {
        "采集流": len(inspection.streams),
        "USD 部件": len(reference["links"]),
        "活动关节": len(reference["joints"]),
        "来源冲突 (已记录)": len(reference["metadata_conflicts"]),
    }
)


## 2. 相机与米制尺度

恢复两组静态采集的相机，用共享标记连接坐标系，再以 USD 高度确定米制尺度。使用所选状态的全部图片恢复相机，并检查是否满足后续训练数量；此步解决长度单位，物体到 USD 的完整刚体位姿由第 4 节的深度表面配准求解。

In [ ]:
camera_operation = px.assets.recover_cameras(
    observations,
    method="colmap",
    config={
        "execution_device": "cpu",
        "matching": "sequential",
        "max_frames_per_stream": None,
        "min_registered_images": 150,
        "stream_names": ["can.start.rgb", "can.end.rgb"],
        "num_threads": 1,
        "random_seed": 0,
    },
)
camera_rig = camera_operation.output("camera_rig_calibration")
observation_artifact = ArtifactRef(
    artifact_id=observations.artifact_id,
    size_bytes=len(px.artifacts.get_bytes(observations.artifact_id)),
    media_type="application/vnd.physrealx.observation-set+json",
)
scale_input = px.artifacts.put_json(
    {
        "schema": "physrealx.model-scale-input/1",
        "observations": observation_artifact.model_dump(mode="json"),
        "reference_model": reference_model.model_dump(mode="json"),
    }
)
metric_operation = px.assets.align_cameras_metric(
    camera_rig, scale_input, method="usd_depth_scale", config={}
)
metric_camera = metric_operation.output("camera_rig_calibration")
show_metrics(
    {
        "注册视角": int(camera_operation.metrics["registered_image_count"]),
        "重投影误差 (px)": camera_operation.metrics["mean_reprojection_error_px"],
        "高度验证相对误差": metric_operation.metrics["max_height_holdout_relative_error"],
    }
)


## 3. 数据集 Mask 与深度增强

绑定全部已注册视图的 RGB、深度、相机和逐帧二值 Mask，像素步长为 2；不按帧数二次采样。ArtModeling 的 `img/`、`depth/` 与 `mask/` 必须使用相同帧号；Notebook 不执行分割。随后补全有邻域支持的小孔；大孔洞与断层继续保留无效标记。


In [ ]:
import numpy as np

calibrated_rgbd = px.assets.prepare_rgbd(
    observations,
    metric_camera,
    stride=2,
    max_frames_per_stream=None,
)
depth_operation = px.assets.enhance_depth(
    calibrated_rgbd, method="rgb_guided_fill", config={"max_rmse_m": 0.02}
)
enhanced_depth = depth_operation.output("enhanced_depth")
rgbd_input = MetricRGBDInput(
    camera_rig=metric_camera, enhanced_depth=enhanced_depth, reference_model=reference_model
)
geometry_input = px.artifacts.put_json(rgbd_input)
show_metrics(
    {
        "数据集 Mask 帧": len(json.loads(px.artifacts.get_bytes(calibrated_rgbd))["frames"]),
        "自监督 RMSE (m)": depth_operation.metrics["holdout_rmse_m"],
    }
)
with zipfile.ZipFile(io.BytesIO(px.artifacts.get_bytes(enhanced_depth))) as archive:
    first = json.loads(archive.read("manifest.json"))["frames"][0]
    depth_image = np.load(io.BytesIO(archive.read(first["depth"]["path"])), allow_pickle=False)
    rgb_image = np.load(
        io.BytesIO(px.artifacts.get_bytes(ArtifactRef.model_validate(first["rgb"]))),
        allow_pickle=False,
    )
    foreground_mask = np.load(
        io.BytesIO(
            px.artifacts.get_bytes(ArtifactRef.model_validate(first["view"]["object_mask"]))
        ),
        allow_pickle=False,
    )
show_rgbd(
    rgb_image, depth_image, OUTPUT_DIR / "rgbd-preview.png", mask=foreground_mask
)


## 4. 几何与关节重建

先以训练视图中的逐帧 Mask 和米制深度点云配准配套 USD 表面，并用 50 张／状态的验证视图门禁结果；再把两个观测状态之间的参考刚体运动转换为 ArtGS 的对称半程初始化。该引导只初始化关节，不替代 RGB、Mask、实测深度和相机驱动的三阶段优化，报告明确标记为模型条件化。

每状态使用 **100 张训练图＋50 张验证图**，数量不足即报错。第二阶段仍从采集数据判断关节类型，第三阶段继续优化轴、位置、运动幅度与几何；第 6 节按统一协议计算关节和部件 CD。方法诊断与参考配准证据均随重建报告发布。参数见 [ArtGS](../../docs/api/artgs.md)。


In [ ]:
reconstruction_operation = px.assets.reconstruct_geometry(
    geometry_input,
    method="artgs",
    config={
        "num_slots": 2,
        "state_streams": ["can.start.rgb", "can.end.rgb"],
        "reference_joint": "e_door_197.joint",
        "training_frames_per_state": 100,
        "seed": 2026,
        "max_stage_seconds": 14400,
    },
)
reconstruction_report_ref = reconstruction_operation.output("geometry_reconstruction_report")
reconstruction_report = json.loads(px.artifacts.get_bytes(reconstruction_report_ref))
assert reconstruction_report["evaluation_scope"] == "held_out_views"
assert reconstruction_report["motion_group_count"] == 2
training_counts = Counter(
    frame.rsplit(".", 1)[0] for frame in reconstruction_report["training_frame_ids"]
)
validation_counts = Counter(
    frame.rsplit(".", 1)[0] for frame in reconstruction_report["validation_frame_ids"]
)
assert training_counts == {"can.start.rgb": 100, "can.end.rgb": 100}
assert validation_counts == {"can.start.rgb": 50, "can.end.rgb": 50}
show_metrics({f"{state} / 训练": count for state, count in training_counts.items()})
show_metrics({f"{state} / 验证": count for state, count in validation_counts.items()})
show_metrics({"运动组": reconstruction_report["motion_group_count"]})
reconstruction_bundle = reconstruction_operation.output("visual_geometry_bundle")


### 三阶段诊断

依次展示双状态粗重建与静/动态匹配、关节类型预测、联合重建的验证视角插值。图像来自项目发布的重建包。


In [ ]:
with zipfile.ZipFile(io.BytesIO(px.artifacts.get_bytes(reconstruction_bundle))) as archive:
    stage_manifest = json.loads(archive.read("diagnostics/stages.json"))
    if stage_manifest.get("schema") != "physrealx.artgs-stage-diagnostics/1":
        raise RuntimeError("阶段诊断版本不匹配。")
    stage_images = {}
    for record in stage_manifest["stages"]:
        preview_path = f"diagnostics/{record['preview']}"
        with Image.open(io.BytesIO(archive.read(preview_path))) as image:
            stage_images[record["name"]] = np.asarray(image.convert("RGB"))
show_image_strip(stage_images, OUTPUT_DIR / "artgs-stage-diagnostics.png")
show_metrics({"原生关节类型": ", ".join(stage_manifest["joint_types"])})


## 5. 位姿绑定与关节校核

复用第 4 节已经通过验证视图门禁的深度—USD 配准，不再重复拟合；这里只将该坐标变换绑定到重建外观并发布注册相机。随后核对 USD 两端构型，供物理结构、质量评估和重建外观共同使用。

In [ ]:
registration_input = px.artifacts.put_json(
    ReferenceRegistrationInput(rgbd=rgbd_input, reconstruction_report=reconstruction_report_ref)
)
registration_operation = px.assets.align_cameras_metric(
    metric_camera,
    registration_input,
    method="articulated_surface_icp",
    config={"joint": "e_door_197.joint"},
)
reference_registration = registration_operation.output("metric_alignment_report")
reference_registration_report = json.loads(px.artifacts.get_bytes(reference_registration))
assert reference_registration_report["fit_stage"] == "pre_reconstruction"
assert reference_registration_report["gates"]["reference_guided_training"] is True
registered_cameras = registration_operation.output("camera_rig_calibration")
joint_positions_closed = {joint["name"]: joint["lower_limit"] for joint in reference["joints"]}
joint_positions_open = {joint["name"]: joint["upper_limit"] for joint in reference["joints"]}
closed_kinematics = evaluate_reference_kinematics(
    reference_model, px.artifacts, joint_positions_closed
)
open_kinematics = evaluate_reference_kinematics(reference_model, px.artifacts, joint_positions_open)
show_metrics(
    {
        "验证视图最大中位距离 (mm)": registration_operation.metrics["max_holdout_median_m"] * 1000,
        "验证视图最大 P90 距离 (mm)": registration_operation.metrics["max_holdout_p90_m"] * 1000,
        "校核关节数": len(reference["joints"]),
    }
)


## 6. 重建质量评估

本次评估烤箱门（`e_door_197`）的部件 CD 与关节指标；箱体（`e_body_203`）参与静态 CD 与整体 CD。

指标统一按 DTArt 计算：CD 单位 **10⁻³ m²**，Axis Ang 单位 **°**，Part Motion 为旋转误差（°）或位移误差（m）；Axis Pos 仅适用于旋转关节，单位 **10⁻¹ m**。当前参考状态来自 USD 配准；评估优先使用方法输出的末态网格，缺失时按声明的预测运动推导，来源随报告记录，属于模型参考评估。

首先展示重建方法发布的全部外观状态帧；随后左侧展示全部重建部件，右侧展示 GT（配套 USD 几何）。几何对比采用同一视角、米制尺度与配准，两侧分别按各自首尾状态插值，箱体保持静止；颜色表示对应运动组。动画展示外观、几何与运动差异，不代表实拍时间或动力学仿真。


In [ ]:
reconstruction_evaluation_ref = px.assets.evaluate_reconstruction(
    px.artifacts.put_json(
        ReconstructionEvaluationInput(
            reconstruction_report=reconstruction_report_ref,
            reference_model=reference_model,
            registration=reference_registration,
            joint_names=("e_door_197.joint",),
        )
    ),
)
show_reconstruction_evaluation(
    json.loads(px.artifacts.get_bytes(reconstruction_evaluation_ref)),
    labels={"e_body_203": "箱体", "e_door_197": "烤箱门"},
)
reconstruction_appearance = px.assets.preview_reconstruction_appearance(
    reconstruction_report_ref
)
show_animation(px.artifacts.get_bytes(reconstruction_appearance))
reconstruction_animation = px.assets.preview_reconstruction(reconstruction_evaluation_ref)
show_animation(px.artifacts.get_bytes(reconstruction_animation))


## 7. 结果与导出

导出下一模块所需的参考结构、重建外观和配准结果。页面保留关键指标与预览，完整方法配置、来源及质量报告保存在 `output/`；必要阶段失败会停止执行。

In [ ]:
exports = {
    "reference-model.json": reference_model,
    "reference-usd.zip": ArtifactRef.model_validate(reference["bundle"]),
    "reference-registration.json": reference_registration,
    "registered-cameras.json": registered_cameras,
    "metric-cameras.json": metric_camera,
    "calibrated-rgbd.json": calibrated_rgbd,
    "enhanced-depth.zip": enhanced_depth,
    "reconstruction-input.json": geometry_input,
    "reconstruction.zip": reconstruction_bundle,
    "reconstruction-report.json": reconstruction_report_ref,
    "reconstruction-evaluation.json": reconstruction_evaluation_ref,
    "reconstruction-appearance.gif": reconstruction_appearance,
    "reconstruction-comparison.gif": reconstruction_animation,
    "kinematics-closed.json": closed_kinematics,
    "kinematics-open.json": open_kinematics,
}
results = {name: ref.model_dump(mode="json") for name, ref in exports.items()}
results_ref = px.artifacts.export(OUTPUT_DIR, results, files=exports)
print("结果索引:", OUTPUT_DIR / "results.json")
